# Analisis Operativo - Procesos de Clientes
Notebook derivado del script `analisis_procesos_clientes.py` para ejecucion por secciones.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import altair as alt

print('Librerias cargadas correctamente')

## 1) Carga de datos

In [ ]:
DATA_PATH = Path('dataset_procesos_clientes_01.xlsx')

def cargar_dataset(ruta: Path) -> pd.DataFrame:
    df = pd.read_excel(ruta)
    df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
    print(f'Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas')
    return df

df_raw = cargar_dataset(DATA_PATH)
print('Tipo de dato de fecha:', df_raw['fecha'].dtype)

## 2) Limpieza y variables operativas

In [ ]:
SLA_POR_PROCESO = {
    'Realizar pagos': 30,
    'Abrir productos': 60,
    'Reportar Fraudes': 45,
    'Realizar reclamaciones': 90,
    'Solicitar soportes o reportes transaccionales': 60,
}

ESTADOS_VALIDOS = {'exitoso', 'con_error'}

def limpiar_datos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=['fecha', 'duracion_minutos'])
    df['estado'] = df['estado'].fillna('desconocido').str.lower().str.strip()
    df = df[df['duracion_minutos'] > 0]

    q1 = df['duracion_minutos'].quantile(0.25)
    q3 = df['duracion_minutos'].quantile(0.75)
    iqr = q3 - q1
    techo = q3 + 1.5 * iqr
    df['duracion_minutos'] = df['duracion_minutos'].clip(upper=techo)

    df['estado_valido'] = df['estado'].isin(ESTADOS_VALIDOS)
    df['con_error'] = (df['estado'] == 'con_error').astype(int)
    df['sla_limite'] = df['tipo_proceso'].map(SLA_POR_PROCESO)
    df['cumple_sla'] = df['duracion_minutos'] <= df['sla_limite']
    return df

def clasificar_riesgo(df: pd.DataFrame) -> pd.DataFrame:
    def regla(row: pd.Series) -> str:
        sla = row['sla_limite'] if pd.notnull(row['sla_limite']) else 60
        if row['con_error'] == 1 and row['duracion_minutos'] > sla * 1.2:
            return 'alto'
        if row['con_error'] == 1 or row['duracion_minutos'] > sla:
            return 'medio'
        return 'bajo'

    df = df.copy()
    df['riesgo'] = df.apply(regla, axis=1)
    return df

df = limpiar_datos(df_raw)
df = clasificar_riesgo(df)
print('Registros procesados:', len(df))

## 3) KPIs y calidad de datos

In [ ]:
def calcular_kpis(df: pd.DataFrame) -> dict:
    estados_validos = df['estado_valido'].sum()
    pct_error = round(df.loc[df['estado_valido'], 'con_error'].mean() * 100, 2) if estados_validos else 0.0
    return {
        'volumen_total': len(df),
        'tiempo_promedio_min': round(df['duracion_minutos'].mean(), 1),
        'pct_error': pct_error,
        'pct_estado_desconocido': round((~df['estado_valido']).mean() * 100, 2),
        'pct_cumplimiento_sla': round(df['cumple_sla'].mean() * 100, 2),
    }

kpis = calcular_kpis(df)
print('KPIs clave')
print(kpis)

print('\nDiagnostico de calidad de datos')
print(df_raw.isnull().sum())

q1_raw = df_raw['duracion_minutos'].dropna().quantile(0.25)
q3_raw = df_raw['duracion_minutos'].dropna().quantile(0.75)
iqr_raw = q3_raw - q1_raw
techo_raw = q3_raw + 1.5 * iqr_raw

print('\nDiagnostico de calidad procesado')
print({
    'registros_con_estado_desconocido': int((~df['estado_valido']).sum()),
    'registros_con_duracion_outlier': int((df_raw['duracion_minutos'].dropna() > techo_raw).sum()),
})

## 4) Visualizaciones y exportacion

In [ ]:
alt.data_transformers.disable_max_rows()
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

Kpi1 = alt.Chart(df).mark_boxplot(extent='min-max', size=40).encode(
    x=alt.X('tipo_proceso:N', title='Tipo de Proceso', axis=alt.Axis(labelAngle=-30, labelLimit=200)),
    y=alt.Y('duracion_minutos:Q', title='Duracion (min)', scale=alt.Scale(zero=False)),
    color=alt.Color('tipo_proceso:N', legend=None, scale=alt.Scale(scheme='tableau10'))
).properties(title='Duracion por tipo de proceso', width=700, height=350)
Kpi1.save(OUTPUT_DIR / 'Kpi1_duracion_por_proceso.html')

error_proceso = df.groupby('tipo_proceso').agg(total=('con_error', 'count'), errores=('con_error', 'sum')).reset_index()
error_proceso['pct_error'] = (error_proceso['errores'] / error_proceso['total'] * 100).round(1)
Kpi2 = alt.Chart(error_proceso).mark_bar().encode(
    x=alt.X('tipo_proceso:N', title='Tipo de Proceso', axis=alt.Axis(labelAngle=-30, labelLimit=200)),
    y=alt.Y('pct_error:Q', title='% Procesos con Error'),
    tooltip=['tipo_proceso:N', 'pct_error:Q', 'errores:Q', 'total:Q']
).properties(title='Tasa de error por proceso', width=700, height=300)
Kpi2.save(OUTPUT_DIR / 'Kpi2_error_por_proceso.html')

sla_resumen = df.groupby('tipo_proceso')['cumple_sla'].mean().mul(100).round(1).reset_index()
Kpi3 = alt.Chart(sla_resumen).mark_bar().encode(
    x=alt.X('cumple_sla:Q', title='% Cumplimiento SLA', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('tipo_proceso:N', title='Tipo de Proceso', sort=alt.EncodingSortField('cumple_sla', order='descending')),
    tooltip=['tipo_proceso:N', alt.Tooltip('cumple_sla:Q', title='% SLA', format='.1f')]
).properties(title='Cumplimiento SLA por proceso', width=550, height=280)
Kpi3.save(OUTPUT_DIR / 'Kpi3_sla_por_proceso.html')

cliente_resumen = df.groupby('id_negocio').agg(
    total_casos=('id_negocio', 'count'),
    interacciones_promedio=('cantidad_interacciones', 'mean'),
    duracion_promedio=('duracion_minutos', 'mean')
).reset_index()
top_clientes = cliente_resumen.sort_values(['total_casos', 'interacciones_promedio'], ascending=[False, False]).head(10)
Kpi4 = alt.Chart(top_clientes).mark_bar().encode(
    x=alt.X('total_casos:Q', title='Total de casos'),
    y=alt.Y('id_negocio:N', title='Cliente / negocio', sort='-x'),
    color=alt.Color('interacciones_promedio:Q', title='Interacciones promedio', scale=alt.Scale(scheme='oranges'))
).properties(title='Carga operativa por cliente', width=700, height=320)
Kpi4.save(OUTPUT_DIR / 'Kpi4_casos_por_cliente.html')

df['mes'] = df['fecha'].dt.month
casospor_mes = df.groupby('mes').size().reset_index(name='total_casos').sort_values('mes')
mapa_meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
casospor_mes['mes_nombre'] = casospor_mes['mes'].map(mapa_meses)
orden_meses = list(mapa_meses.values())

base_kpi5 = alt.Chart(casospor_mes).encode(
    x=alt.X('mes_nombre:N', sort=orden_meses, title='Mes', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('total_casos:Q', title='Cantidad de casos'),
    tooltip=[alt.Tooltip('mes_nombre:N', title='Mes'), alt.Tooltip('total_casos:Q', title='Casos', format=',')]
)
bars_kpi5 = base_kpi5.mark_bar(size=34, cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    color=alt.Color('total_casos:Q', legend=None, scale=alt.Scale(scheme='tealblues'))
)
labels_kpi5 = base_kpi5.mark_text(dy=-8, fontSize=11).encode(text=alt.Text('total_casos:Q', format=','))
Kpi5 = (bars_kpi5 + labels_kpi5).properties(title='Volumen de casos por mes', width=680, height=320)
Kpi5.save(OUTPUT_DIR / 'Kpi5_casos_por_mes.html')

print('Visualizaciones generadas en outputs/')

## 5) Insights y recomendaciones

In [ ]:
mes_max = casospor_mes.sort_values('total_casos', ascending=False).iloc[0]
tiempo_por_proceso = df.groupby('tipo_proceso')['duracion_minutos'].mean().sort_values(ascending=False)
error_por_proceso = error_proceso.set_index('tipo_proceso')['pct_error'].sort_values(ascending=False)
sla_por_proceso = sla_resumen.set_index('tipo_proceso')['cumple_sla'].sort_values()
responsable_casos = df['responsable'].value_counts()
cliente_top = top_clientes.iloc[0]

print('INSIGHTS')
print(f'1) Mayor duracion promedio: {tiempo_por_proceso.index[0]} ({tiempo_por_proceso.iloc[0]:.1f} min)')
print(f'2) Mayor tasa de error: {error_por_proceso.index[0]} ({error_por_proceso.iloc[0]:.1f}%)')
print(f'3) Cliente con mayor carga operativa: {cliente_top["id_negocio"]} ({int(cliente_top["total_casos"])} casos)')
print(f'4) Responsable con mas casos: {responsable_casos.index[0]} ({responsable_casos.iloc[0]} casos)')
print(f'5) Menor cumplimiento SLA: {sla_por_proceso.index[0]} ({sla_por_proceso.iloc[0]:.1f}%)')
print(f'6) Pico de demanda: {mes_max["mes_nombre"]} ({int(mes_max["total_casos"])} casos)')

print('\nRECOMENDACIONES PRIORIZADAS')
print('1) Priorizar el proceso con mayor duracion y/o mayor error para rediseno operativo.')
print('2) Corregir captura de estados desconocidos para fortalecer los KPIs de calidad.')
print('3) Aplicar estrategias de autoservicio para clientes con mayor carga operativa.')
print('4) Planificar capacidad segun estacionalidad mensual.')